In [14]:
import pandas as pd
import json
from pathlib import Path
from collections import Counter

Load data from directory

In [15]:
file_path = Path("Data/raw/preprocessed_capstone2025.json")
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(len(data)) #check for successful load

53730


In [16]:
records = []
for doc_id, content in data.items():
    base = {
        "doc_id": doc_id.removesuffix(".xml"),
        "schema": content.get("schema"),
        "dokument_art": content.get("dokument_art", [None])[0],
        "dokument_typ": content.get("dokument_typ", [None])[0],
        "institution": content.get("bibliographische-angaben", {}).get("institution", [None])[0],
        "aktenzeichen": content.get("bibliographische-angaben", {}).get("aktenzeichen", [None])[0],
        "datum": content.get("bibliographische-angaben", {}).get("datum", [None])[0],
        "fundstelle": "; ".join(content.get("bibliographische-angaben", {}).get("fundstelle_kuerzel", [])),
        "vorinstanz": "; ".join(content.get("bibliographische-angaben", {}).get("vorinstanz", [])),
        "norm_vorinstanz_aktenzeichen": "; ".join(content.get("bibliographische-angaben", {}).get("norm_vorinstanz_aktenzeichen", [])),
        "norm_kuerzel": "; ".join(content.get("bibliographische-angaben", {}).get("norm_kuerzel", [])),
        "titel": content.get("text", {}).get("titel", [None])[0],
        "rthema": "; ".join(content.get("allgemeine-angaben", {}).get("rthema", [])),
        "quelle": content.get("interne-angaben", {}).get("quelle", [None])[0]
    }
    records.append(base)

# Create the DataFrame
df = pd.DataFrame(records)

# Show how many documents loaded
print(f"Loaded {len(df)} documents.")
# Check the first few row of the DataFrame
print(df.columns)

Loaded 53730 documents.
Index(['doc_id', 'schema', 'dokument_art', 'dokument_typ', 'institution',
       'aktenzeichen', 'datum', 'fundstelle', 'vorinstanz',
       'norm_vorinstanz_aktenzeichen', 'norm_kuerzel', 'titel', 'rthema',
       'quelle'],
      dtype='object')


Select only documents from 2000 to 2020

In [17]:
# First, make sure 'datum' is parsed as datetime
df["datum"] = pd.to_datetime(df["datum"], errors="coerce")  # Turn invalid dates into NaT (Not a Time)

# Now filter the DataFrame for dates between 2000 and 2020
mask = (df["datum"] >= "2000-01-01") & (df["datum"] <= "2020-12-31")
df_year_filtered = df.loc[mask]

# Show how many documents remain
print(f"After filtering for documents from 2000 until 2020 , {len(df_year_filtered)} documents remain.")

After filtering for documents from 2000 until 2020 , 43625 documents remain.


Select only Urteile from 2000 to 2020  (dokument_art)

In [18]:
# filter for documents where 'dokument_art' is 'Urteil'
df_filtered_art = df_year_filtered[df_year_filtered["dokument_art"] == "Urteil"]

# Show how many documents remain after this second filter
print(f"After filtering for 'Urteil', {len(df_filtered_art)} documents remain that are from 2000 to 2020.")


After filtering for 'Urteil', 14186 documents remain that are from 2000 to 2020.


Select only Urteile from 2000 to 2020 that are Obere Rechtsprechung. 


In [19]:
# filter for documents where 'dokument_type' is not 'Literatur'
df_filtered_typ = df_filtered_art[df_filtered_art["dokument_typ"] == "Obere Rechtsprechung"]

# Show how many documents remain after this second filter
print(f"After filtering for 'Obere Rechtsprechung', {len(df_filtered_typ)} documents remain that are from 2000 to 2020.")


After filtering for 'Obere Rechtsprechung', 14182 documents remain that are from 2000 to 2020.


Filter for rthema that should possibly be included

In [ ]:
# define  allowed rthema list
allowed_rthema = [
    "Zivilverfahrensrecht", "SchuldrechtAT", "Schadensersatz",
    "Handelsrecht Gesellschaftsrecht", "BGBAT", "Miete Pacht",
    "Sachenrecht", "Familienrecht", "Versicherungsrecht",
    "Kauf Tausch Leasing", "Erbschaft Schenkung", "EDV-Recht",
    "Schuldverhältnisse", "Werkvertrag", "EU-Recht", "Wohnungseigentum",
    "Sonstiges Recht", "Reisevertrag", "HandelsrechtGesellschaftsrecht", "MietePacht"
]

# Define a function to check if all rthema are in the allowed list
def all_rthema_allowed(rthema_string):
    if pd.isna(rthema_string):
        return False  # Exclude documents with missing rthema
    rthema_list = [r.strip() for r in rthema_string.split(";")]
    return all(r in allowed_rthema for r in rthema_list)

# Apply the function to filter the dataset
df_filtered_rthema_strict = df_filtered_typ[df_filtered_typ["rthema"].apply(all_rthema_allowed)]

# Reset index after filtering
df_filtered_rthema_strict = df_filtered_rthema_strict.reset_index(drop=True)

# Show the result
print(f"After strict rthema filtering, {len(df_filtered_rthema_strict)} documents remain.")


After strict rthema filtering, 7124 documents remain.


If exclude the optional topics: Familienrecht und EDV-Recht

In [21]:
# define  allowed rthema list
allowed_rthema_small = [
    "Zivilverfahrensrecht", "SchuldrechtAT", "Schadensersatz",
    "Handelsrecht Gesellschaftsrecht", "BGBAT", "Miete Pacht",
    "Sachenrecht", "Versicherungsrecht",
    "Kauf Tausch Leasing", "Erbschaft Schenkung", 
    "Schuldverhältnisse", "Werkvertrag", "EU-Recht", "Wohnungseigentum",
    "Sonstiges Recht", "Reisevertrag", "HandelsrechtGesellschaftsrecht", "MietePacht"
]

# Define a function to check if all rthema are in the allowed list
def all_rthema_allowed(rthema_string):
    if pd.isna(rthema_string):
        return False  # Exclude documents with missing rthema
    rthema_list = [r.strip() for r in rthema_string.split(";")]
    return all(r in allowed_rthema_small for r in rthema_list)

# Apply the function to filter the dataset
df_filtered_rthema_strict = df_filtered_typ[df_filtered_typ["rthema"].apply(all_rthema_allowed)]

# Reset index after filtering
df_filtered_rthema_strict = df_filtered_rthema_strict.reset_index(drop=True)

# Show the result
print(f"After removing also EDV-Recht and Familienrecht, {len(df_filtered_rthema_strict)} documents remain.")

After removing also EDV-Recht and Familienrecht, 6822 documents remain.
